## 1. The mean and the median
Load the table. First, make a small `pd.Series` with five number sof reviews that are close to each other, and calculate the mean and the median- Then add one listing with 1293 reviews (themost popular one) and calculate again. What happened to each number? Then calculate the mean and the median of the real column *number_of_reviews* for all 4,574 listings. Which one would you use to answer "how many reviews does a typical listing have?", and why?

In [1]:
import pandas as pd

listings_raw = pd.read_csv("data/listings_clean.csv", parse_dates=["first_review",
                                                                   "last_review",
                                                                   "last_scraped",
                                                                   "calendar_last_scraped",
                                                                   "price_quote_checkin_date",
                                                                   "price_quote_checkout_date"])


listings = listings_raw.copy()

# 4574 rows and 73 columns
print(listings.shape)

five = pd.Series([3, 5, 8, 10, 12])

# mean is 7.6, median is 8
print(five.mean())
print(five.median())

six = pd.Series([3, 5, 8, 10, 12, 1293])

# mean is now 221.83, median is 9
print(six.mean())
print(six.median())

# mean is 35.75, median is 9
print(listings["number_of_reviews"].mean())
print(listings["number_of_reviews"].median())

# i would use median since a outstanding number doesn't change the median
# as much as it changes the mean

(4574, 73)
7.6
8.0
221.83333333333334
9.0
35.75404459991255
9.0


## 2. Percentiles and the spread
Run `describe()` on *number_of_reviews*. Then calculate the 10th, 25th, 50th, 75th, 90th, and 99th percentile with `quantile`. Calculate *q1*, *q3* and the *iqr*. Write one sentence in your own words about what the IQR says here. Look at the 10th percentile: what does it tell you about how many listings have no reviews at all?

In [2]:
print(listings["number_of_reviews"].describe())

print(listings["number_of_reviews"].quantile([0.1, 0.25, 0.5, 0.75, 0.90, 0.99]))

q1 = listings["number_of_reviews"].quantile(0.25)
q3 = listings["number_of_reviews"].quantile(0.75)

iqr = q3 - q1

# q1 is 2, q3 is 29, iqr is 27
print(q1)
print(q3)
print(iqr)

# 50% of the listings have between 2 and 29 reviews
# at least 10% of the listings dont have reviews

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64
0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64
2.0
29.0
27.0


## 3. The fences and the outliers
Calculate the upper and the lower fence with the IQR rule. Can there be outliers below the lower fence for this column? Why, or why not? How many listings are above the upper fence? Then look at them: how many of each room type, and what is their median *price* compared with all the other listings? Write what you see in a comment.

In [3]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5* iqr

# upper is 69.5, lower is -38.5
print(upper)
print(lower)

# there can't be any outlier below lower, since it's negative,
# and number_of_reviews can't be negative

outliers = listings[listings["number_of_reviews"] > upper]

print(len(outliers))
print(outliers["number_of_reviews"].describe())

# Entire home/apt is 446
# Private room is 121
# Shared room is 14
# Hotel room is 1
print(outliers["room_type"].value_counts())

# outliers are cheaper and most are Entire home/apt (1459.54 vs 1867)

# median is 1459.54
print(outliers["price"].median())

others = listings[listings["number_of_reviews"] <= upper]

# median is 1867
print(others["price"].median())

69.5
-38.5
582
count     582.000000
mean      192.187285
std       136.400813
min        70.000000
25%        98.000000
50%       148.500000
75%       236.500000
max      1293.000000
Name: number_of_reviews, dtype: float64
room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64
1459.54
1867.0


## 4. Mark them, and compare
Do not remove the outliers. Mark them in a new column, *reviews_outlier* (True or False), and count them. Then make a table with `pd.DataFrame` that shows *count*, *mean*, *median* and *std* for *number_of_reviews*, with all listings and without the outliers. How many percent does each number change when the outliers are gone? Which changes the most, and which the least?

In [4]:
listings["reviews_outlier"] = listings["number_of_reviews"] > upper

# 582 are True, 3992 are False
print(listings["reviews_outlier"].value_counts())

compare = pd.DataFrame({
    "all": listings["number_of_reviews"].agg(["count", "mean", "median", "std"]),
    "without_outliers": listings.loc[~listings["reviews_outlier"], "number_of_reviews"].agg(["count", "mean", "median", "std"])
})

# std changes the most (-80) and median changes the least (-22)
print(compare)
print(((compare["without_outliers"] - compare["all"]) / compare["all"] * 100).round(0))

reviews_outlier
False    3992
True      582
Name: count, dtype: int64
                all  without_outliers
count   4574.000000       3992.000000
mean      35.754045         12.947395
median     9.000000          7.000000
std       78.377200         15.537791
count    -13.0
mean     -64.0
median   -22.0
std      -80.0
dtype: float64
